# 02 Data Cleaning
Clean text data, handle missing values, remove duplicates, and prepare for analysis.

In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from pathlib import Path

# Download NLTK data
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# Paths
DATA_PATH = '../data/Amazon_Reviews.csv'
OUTPUT_PATH = '../data/cleaned_reviews.csv'

print('Loading raw data...')
df = pd.read_csv(DATA_PATH, engine="python")
print(f'Initial shape: {df.shape}')

Loading raw data...
Initial shape: (21214, 9)


In [6]:
# Text cleaning function
def clean_text(text):
    if pd.isna(text):
        return ''
    # lowercase
    text = str(text).lower()
    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # remove HTML entities
    text = re.sub(r'&[a-z]+;', '', text)
    # remove special characters, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # remove stopwords and short tokens
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

print('Cleaning text...')
if 'Review Text' in df.columns:
    df['reviewText_clean'] = df['Review Text'].apply(clean_text)
    df = df[df['reviewText_clean'].str.strip().astype(bool)].reset_index(drop=True)
    
print(f'After cleaning: {df.shape}')

Cleaning text...
After cleaning: (21054, 10)


In [7]:
# Remove duplicates
print('Removing duplicates...')
initial = len(df)
df = df.drop_duplicates(subset=['reviewText_clean'], keep='first').reset_index(drop=True)
removed = initial - len(df)
print(f'Removed {removed} duplicates. Shape: {df.shape}')

Removing duplicates...
Removed 684 duplicates. Shape: (20370, 10)


In [10]:
print(df['Rating'].head())
print(df['Rating'].dtype)

0    Rated 1 out of 5 stars
1    Rated 1 out of 5 stars
2    Rated 1 out of 5 stars
3    Rated 1 out of 5 stars
4    Rated 1 out of 5 stars
Name: Rating, dtype: str
str


In [11]:
df['Rating_Num'] = (
    df['Rating']
    .str.extract(r'(\d+)')[0]
    .astype(float)
)

In [12]:
print(df['Rating_Num'].head())

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: Rating_Num, dtype: float64


In [15]:
# Handle missing values
print('Handling missing values...')
if 'Review Title' in df.columns:
    df['Review Title'] = df['Review Title'].fillna('No title')
if 'Rating_Num' in df.columns:
    df['Rating_Num'] = df['Rating_Num'].fillna(df['Rating_Num'].median())

# Create sentiment label (binary)
def get_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

df["sentiment"] = df["Rating_Num"].apply(get_sentiment)

print(df["sentiment"].value_counts())

print(f'\nFinal shape: {df.shape}')

Handling missing values...
sentiment
Negative    14150
Positive     5398
Neutral       822
Name: count, dtype: int64

Final shape: (20370, 12)


In [16]:
# Save cleaned dataset
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f'Cleaned data saved to {OUTPUT_PATH}')

# Display sample
df[['reviewText_clean', 'Rating_Num', 'sentiment']].head(10)

Cleaned data saved to ../data/cleaned_reviews.csv


,reviewText_clean,Rating_Num,sentiment
0,registered website tried order laptop entered ...,1.0,Negative
1,multiple orders one turned driver phone door n...,1.0,Negative
2,informed reprobates would going visit sick rel...,1.0,Negative
3,bought amazon problems happy service price ama...,1.0,Negative
4,could give lower rate would cancelled amazon p...,1.0,Negative
5,terrible get customer service reps clearly hom...,1.0,Negative
6,amazon way tainting great product due inabilit...,1.0,Negative
7,love amazon use half shopping prime membership...,5.0,Positive
8,applied job amazon completed steps including s...,1.0,Negative
9,great experience customer service delivered or...,5.0,Positive
